###  excel替换数据源（日常分析、周报使用） 
#### 登陆阿里云，读取数据

In [ ]:
import os
from odps import ODPS
import numpy as np
import pandas as pd

##initialize odps
o = ODPS(
    # （推荐）确保已设置环境变量。
    # 确保ALIBABA_CLOUD_ACCESS_KEY_ID环境变量设置为用户 Access Key ID。
    access_id=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_ID'),
    
    # 确保ALIBABA_CLOUD_ACCESS_KEY_SECRET环境变量设置为用户Access Key Secret。
    secret_access_key=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_SECRET'),
    project='xyf_jingying_dev',
    endpoint='https://service.cn-beijing.maxcompute.aliyun.com/api',
)

import shutil
from openpyxl import load_workbook
from openpyxl.styles import Font, Fill, Border, Alignment, Side, PatternFill
from win32com.client import Dispatch
from datetime import datetime , timedelta 

# 自定义bizdate 与数据统计区间
bizdate = (datetime.now() - timedelta(days=1)).strftime('%Y%m%d')  # 业务日期设置为当前日期的昨天
begin_date = (datetime.now() - timedelta(days=60)).strftime('%Y-%m-%d') # 开始日期设置为当前日期的前60天
end_date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')  # 结束日期设置为当前日期的昨天

In [ ]:
## SQL查询
query='''
SELECT  DATE(loan_time)                                                 AS 放款日期
       ,SUM(loan_amt)                                                   AS 总放款
       ,SUM(CASE WHEN asset_type_flag = 'I24' THEN loan_amt ELSE 0 END) AS 24放款
       ,SUM(CASE WHEN asset_type_flag = 'I36' THEN loan_amt ELSE 0 END) AS 36放款
FROM
(
	SELECT  loan_time
	       ,loan_amt
	       ,period
	       ,asset_type_flag
	       ,fee_rate
	       ,first_order_number
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = '{bizdate}'
	AND DATE(first_order_time) >= '{begin_date}'
    AND DATE(first_order_time) <= '{end_date}'
	AND loan_time IS NOT NULL 
) a
GROUP BY  DATE(loan_time)
ORDER BY  DATE(loan_time)
'''.format(bizdate = bizdate, begin_date = begin_date, end_date = end_date)

In [ ]:
#放款统计
result = o.execute_sql(query)
loan_stats = result.open_reader().to_pandas()

# ===========================================
# 放款统计数据格式化
# ===========================================
# 日期和字符串字段
loan_stats['放款日期'] = pd.to_datetime(loan_stats['放款日期']).dt.strftime('%Y-%m-%d')

# 浮点数字段 - 保留2位小数
f_columns_loan = ['总放款', '24放款', '36放款']
for col in f_columns_loan:
    if col in loan_stats.columns:
        loan_stats[col] = loan_stats[col].astype('float64')

# 整数字段
int_columns_loan = []
for col in int_columns_loan:
    if col in loan_stats.columns:
        loan_stats[col] = loan_stats[col].astype('int64')

loan_stats

#### 替换底表数据，建议数据源根据名称管理器设置成动态数据透视表以方便后续刷新扩展数据

In [ ]:
def write_dataframe_to_excel_com(file_path, dataframes_dict, start_row=1, start_col=1, include_header=True):
    """
    Args:
        file_path: Excel文件路径
        dataframes_dict: {sheet_name: dataframe} 字典
        start_row: 从第几行开始写入（默认1）
        start_col: 从第几列开始写入（默认1）
        include_header: 是否写入表头（默认True）
    """
    # 启动Excel应用
    excel_app = Dispatch("Excel.Application")
    excel_app.Visible = False               # 后台运行
    excel_app.DisplayAlerts = False         # 禁用警告对话框

    try:
        # 打开工作簿
        workbook = excel_app.Workbooks.Open(file_path)

        for sheet_name, df in dataframes_dict.items():
            try:
                # 获取工作表
                worksheet = workbook.Worksheets(sheet_name)

                # 连表头/旧内容一起清空
                used = worksheet.UsedRange
                if used is not None and used.Rows.Count > 0 and used.Columns.Count > 0:
                    used.ClearContents()

                if df is not None and not df.empty:
                    # 要写入的数据：可选表头 + 数据
                    if include_header:
                        values = [df.columns.tolist()] + df.values.tolist()
                    else:
                        values = df.values.tolist()

                    n_rows = len(values)
                    n_cols = len(values[0]) if n_rows > 0 else 0

                    end_row = start_row + n_rows - 1
                    end_col = start_col + n_cols - 1

                    write_range = worksheet.Range(
                        worksheet.Cells(start_row, start_col),
                        worksheet.Cells(end_row, end_col),
                    )
                    write_range.Value = values

                print(f"成功写入工作表: {sheet_name}")

            except Exception as e:
                print(f"写入工作表 {sheet_name} 时出错: {e}")
                continue
        
        workbook.RefreshAll()                       # 刷新所有数据连接和透视表
        excel_app.Calculate()                       # 强制重新计算公式
        excel_app.CalculateUntilAsyncQueriesDone()  # 等待所有操作完成
        
        # 保存并关闭
        workbook.Save()
        workbook.Close()
        print(f"文件已保存: {file_path}")

    except Exception as e:
        print(f"操作Excel文件时出错: {e}")

    finally:
        excel_app.Quit()        # 退出Excel应用

In [ ]:
file_path = r"D:\python自动化实践\excel样例.xlsx"
write_dataframe_to_excel_com(file_path, {"放款统计": loan_stats}, start_row=1, include_header=True)

#### 批量添加计算字段（有待进一步优化，目前首次添加计算字段无问题，重复添加有问题待优化）

In [ ]:
def add_pivot_calculated_fields_com(
    file_path: str,
    sheet_name: str,
    pivot_name: str | None = None,
    pivot_index: int = 1,
    fields: dict | None = None,
    refresh: bool = True,
    add_to_values: bool = True,          # 是否自动加入“值”区域
    skip_if_exists: bool = True,         # 同名计算字段已存在时：True=跳过；False=尝试删除后重建（不稳定）
    verbose: bool = True,                # 是否打印过程日志
):
    """
    在已有 PivotTable 上批量添加 Calculated Fields（计算字段）
    - 可选：自动加入值区域
    - 若遇到同名计算字段/异常：记录并跳过，继续执行剩余字段
    - 执行结束：print 未成功字段清单

    fields 格式：
    {
      "金额加权平均定价2": {"formula": "='金额*定价(%)'/放款金额", "number_format": "0.00", "caption": "金额加权平均定价2"},
      ...
    }
    """
    if not fields:
        raise ValueError("fields 不能为空，例如：{'字段名': {'formula': '=<expr>', 'number_format': '0.00%'}}")

    failed = []  

    excel_app = Dispatch("Excel.Application")
    excel_app.Visible = False
    excel_app.DisplayAlerts = False

    # 常量
    xlDataField = 4  # PivotField 作为“值”区域

    try:
        wb = excel_app.Workbooks.Open(file_path)
        ws = wb.Worksheets(sheet_name)

        # 选取 PivotTable（建议用 pivot_name）
        pvt = ws.PivotTables(pivot_name) if pivot_name else ws.PivotTables(pivot_index)

        if verbose:
            try:
                print(f"[Pivot] sheet={sheet_name} pivot={pvt.Name} cache_index={pvt.PivotCache().Index}")
            except Exception:
                print(f"[Pivot] sheet={sheet_name}")

        for field_name, cfg in fields.items():
            cfg = cfg or {}
            formula = cfg.get("formula")
            if not formula:
                failed.append({"field": field_name, "reason": "missing_formula", "detail": "cfg 中缺少 formula"})
                continue

            number_format = cfg.get("number_format", "General")
            caption = cfg.get("caption", field_name)

            # 1) 检查同名计算字段是否已存在
            exists = False
            try:
                _ = pvt.CalculatedFields(field_name)  # 存在则不报错
                exists = True
            except Exception:
                exists = False

            if exists and skip_if_exists:
                failed.append({"field": field_name, "reason": "already_exists", "detail": "计算字段已存在，按配置跳过"})
                if verbose:
                    print(f"[Skip] {field_name} 已存在，跳过")
                continue

            # 2) （可选）尝试删除旧计算字段（注意：删除常因被其他透视表引用而失败）
            if exists and not skip_if_exists:
                try:
                    pvt.CalculatedFields(field_name).Delete()
                except Exception as e:
                    failed.append({"field": field_name, "reason": "delete_failed", "detail": str(e)})
                    if verbose:
                        print(f"[Fail] 删除旧计算字段失败: {field_name} | {e}")
                    continue

            # 3) 新增计算字段
            try:
                pvt.CalculatedFields().Add(Name=field_name, Formula=formula)
                if verbose:
                    print(f"[OK] Add CalculatedField: {field_name} | {formula}")
            except Exception as e:
                failed.append({"field": field_name, "reason": "add_failed", "detail": str(e)})
                if verbose:
                    print(f"[Fail] Add CalculatedField 失败: {field_name} | {e}")
                continue

            # 4) 是否加入“值区域”
            if add_to_values:
                try:
                    # 更直接：把 PivotField 放到值区域（兼容性更高）
                    pf = pvt.PivotFields(field_name)
                    pf.Orientation = xlDataField
                    pf.NumberFormat = number_format
                    # 如果你想控制显示名，可在 Excel 里再改；COM 里改 Name 有时会触发异常
                    if verbose:
                        print(f"[OK] Add to Values: {field_name} | format={number_format}")
                except Exception as e:
                    # 加值区域失败不影响“计算字段已创建”，但按你的要求也算未成功设置
                    failed.append({"field": field_name, "reason": "add_to_values_failed", "detail": str(e)})
                    if verbose:
                        print(f"[Fail] 加入值区域失败: {field_name} | {e}")
                    continue

        # 5) 刷新（尽量只刷新当前透视表，避免 RefreshAll 造成锁/异步）
        if refresh:
            try:
                pvt.RefreshTable()
            except Exception:
                try:
                    wb.RefreshAll()
                    excel_app.Calculate()
                    try:
                        excel_app.CalculateUntilAsyncQueriesDone()
                    except Exception:
                        pass
                except Exception:
                    pass

        wb.Save()
        wb.Close()

    finally:
        excel_app.Quit()

    # 汇总输出
    if failed:
        print("\n===== 未成功设置的计算字段（含跳过项）=====")
        for item in failed:
            print(f"- {item.get('field')} | {item.get('reason')} | {item.get('detail')}")
        print("===== 结束 =====\n")
    else:
        print("\n全部计算字段设置成功。\n")

    return failed

In [ ]:
file_path = r"D:\python自动化实践\excel样例.xlsx"

calc_fields = {
    "24占比": {"formula": "='24放款'/总放款", "number_format": "0.00%"},
    "36占比": {"formula": "='36放款'/总放款", "number_format": "0.00%"},
}

failed = add_pivot_calculated_fields_com(
    file_path=file_path,
    sheet_name="Sheet1",
    pivot_name="数据透视表1",
    fields=calc_fields,
    refresh=True,
    add_to_values=True,      # 自动加入值区域
    skip_if_exists=True,     # 同名就记录并跳过（避免第二次 Add “发生意外”）
    verbose=True,
)

###  PPT图片替换流程

#### 函数预准备

In [ ]:
import os
from odps import ODPS
import numpy as np
import pandas as pd

##initialize odps
o = ODPS(
    # （推荐）确保已设置环境变量。
    # 确保ALIBABA_CLOUD_ACCESS_KEY_ID环境变量设置为用户 Access Key ID。
    access_id=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_ID'),
    
    # 确保ALIBABA_CLOUD_ACCESS_KEY_SECRET环境变量设置为用户Access Key Secret。
    secret_access_key=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_SECRET'),
    project='xyf_jingying_dev',
    endpoint='https://service.cn-beijing.maxcompute.aliyun.com/api',
)

from pptx import Presentation
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import PercentFormatter
import os

plt.rcParams['font.sans-serif'] = ['STKaiti', 'Kaiti']  # 设置中文为华文楷体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方块
mpl.rcParams.update({
    # 全局默认字号=14（不单独设置 titlesize/legend 等时，会继承 font.size）
    "font.size": 14,
    # 全局关闭 y 轴刻度线 + 刻度文字（等价于每张图都做 ax.tick_params + ax.set_yticklabels([])）
    "ytick.left": False,       # 不画左侧刻度线
    "ytick.right": False,      # 不画右侧刻度线（防止 twinx 时冒出来）
    "ytick.labelleft": False,  # 不画左侧刻度文字
    "ytick.labelright": False, # 不画右侧刻度文字
})
# mpl.rcdefaults()   # 恢复到 matplotlib 默认 rcParams

## 自定义bizdate 与数据统计区间
bizdate = '20251218'
begin_date = '2025-01-01'   
end_date = '2025-12-18'
ppt_path=r'D:\python自动化实践\PPT模版.pptx'
out_path=r'D:\python自动化实践\PPT模版_更新.pptx'

#### 堆积柱状图函数

In [ ]:
# 堆积柱状图函数
def plot_stacked_bar_from_pivot(
    pivot: pd.DataFrame,
    cols: list,
    title: str,
    save_dir: str = None,
    suffix: str = ".png",
    colors: dict = None,
    width: float = 0.55,
    p: float = 1.0,
    fontsize: int = 14,
    top_label_fmt: str = None,
    segment_label_fmt: str = "{p:.1f}%",
    y_label: str = None,
    x_rotation: int = 0,
    dpi: int = 300,
    bbox_inches: str = "tight",
    hide_y_ticks: bool = True,          # y轴不显示刻度线
    hide_y_tick_labels: bool = True,   # 仅隐藏刻度线时一般够用；想更“干净”可设 True
    spine_off: tuple = ("top", "right", "left"),
):
    """
    从 pivot_table 画堆积柱状图，并在每段显示占比、柱顶显示总数。

    参数
    - pivot: DataFrame，index为x轴（如月份），columns为分组，values为数值
    - cols: 你希望的统一列顺序（缺列自动补0）
    - title: 图标题，同时用于保存文件名（title + suffix）
    - save_dir: 保存目录；None 则默认当前工作目录下的 imgs
    - colors: dict，{col: color}。不传则用默认低饱和配色（最多6个）
    - width: 柱宽
    - p: 段内百分比展示阈值（单位：百分比，比如0.5表示<0.5%不标）
    - fontsize: 标签字号（段内 & 顶部）
    - top_label_fmt: 顶部总数格式化字符串，如 "{t:.1f}"；None 则自动根据数据大小选
    - segment_label_fmt: 段内占比格式（百分比），默认"{p:.1f}%"
    """

    if pivot is None or len(pivot) == 0:
        raise ValueError("pivot 为空，无法绘图。")

    # 默认保存目录
    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), "imgs")
    os.makedirs(save_dir, exist_ok=True)

    # 默认配色方案（6个）
    default_palette = [
        "#2F6EEB",  # 蓝
        "#F39C12",  # 橙
        "#9e9e9e",  # 灰
        "#F1C40F",  # 黄
        "#34B4F4",  # 浅蓝
        "#6A5ACD",  # 紫 
    ]

    # 统一列：缺的补0，多的丢弃
    pv = pivot.copy()
    for c in cols:
        if c not in pv.columns:
            pv[c] = 0.0
    pv = pv[cols].copy()

    # 强制数值化
    for c in cols:
        pv[c] = pd.to_numeric(pv[c], errors="coerce").fillna(0.0)
    pv = pv.sort_index()

    totals = pv.sum(axis=1).astype(float)
    pct = pv.div(totals.replace(0, np.nan), axis=0).fillna(0.0) * 100

    # 颜色映射：先用默认，再被用户 colors 覆盖
    color_map = {}
    for i, c in enumerate(cols):
        color_map[c] = default_palette[i % len(default_palette)]
    if colors:
        color_map.update(colors)

    # 顶部总和标签格式：默认智能一点
    if top_label_fmt is None:
        max_total = float(np.nanmax(totals.values)) if len(totals) else 0.0
        # 大多数你这里是“亿/万”这种已经缩放过的值，保留1位小数更常见；否则整数
        top_label_fmt = "{t:.1f}" if max_total < 1000 else "{t:.0f}"

    fig, ax = plt.subplots(figsize=(13.33, 7.5))
    x = np.arange(len(pv.index))
    bottom = np.zeros(len(pv.index), dtype=float)

    # 统一的白底黑字框
    bbox_kw = dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="white", alpha=0.75)

    # 堆积柱
    for c in cols:
        vals = pv[c].values.astype(float)
        ax.bar(x, vals, width, bottom=bottom, label=c, color=color_map.get(c))

        # 段内百分比标签（黑色 + 白底框）
        pct_vals = pct[c].values
        y_center = bottom + vals / 2
        for i, (xc, yc, percent, v) in enumerate(zip(x, y_center, pct_vals, vals)):
            if v > 0 and percent >= p:
                ax.text(
                    xc, yc,
                    segment_label_fmt.format(p=float(percent)),
                    ha="center", va="center",
                    fontsize=fontsize,
                    fontweight="bold",
                    color="black",
                    bbox=bbox_kw,
                )

        bottom += vals

    # 柱顶总数标签（黑色）
    for xc, t in zip(x, totals.values):
        if np.isfinite(t) and t != 0:
            # 让顶部标签稍微上移一点
            offset = max(float(np.nanmax(totals.values)) * 0.01, 0.01)
            ax.text(
                xc, t + offset,
                top_label_fmt.format(t=float(t)),
                ha="center", va="bottom",
                fontsize=fontsize + 1,
                fontweight="bold",
                color="black",
            )

    # 标题/坐标
    ax.set_title(title, fontsize=fontsize + 6, fontweight="bold", pad=20)
    if y_label:
        ax.set_ylabel(y_label)

    ax.set_xticks(x)
    ax.set_xticklabels(pv.index.astype(str), rotation=x_rotation, fontsize=fontsize)

    # y轴：不需要刻度线
    if hide_y_ticks:
        ax.tick_params(axis="y", length=0)
    if hide_y_tick_labels:
        ax.set_yticklabels([])

    # 去边框（保持一致风格）
    for sp in spine_off:
        ax.spines[sp].set_visible(False)

    # 图例放底部居中
    ax.legend(ncol = len(cols), loc="lower center", bbox_to_anchor=(0.5, -0.15), frameon=False, fontsize=fontsize)

    plt.tight_layout()

    # 保存：标题=文件名
    safe_name = title.replace("/", "_").replace("\\", "_").strip()
    img_path = os.path.join(save_dir, f"{safe_name}{suffix}")
    fig.savefig(img_path, dpi=dpi, bbox_inches=bbox_inches)
    plt.show()
    plt.close(fig)

#### 折线图函数

In [ ]:
def plot_lines_from_df(
    df: pd.DataFrame,
    x_col: str,
    y_cols: list,
    title: str,
    *,
    labels: dict = None,                 # {y_col: 显示名}
    colors: dict = None,                 # {y_col: color} 覆盖默认色
    linestyles: dict = None,             # {y_col: '-', ':' ...}
    markers: dict = None,                # {y_col: 'o', ...}
    offsets: dict = None,                # {y_col: (dx, dy)} 用于数据标签偏移（单位：points）
    aligns: dict = None,                 # {y_col: {"ha":"center","va":"bottom"}} 或 {y_col: (ha, va)}
    label_cfg: dict = None,              # 精细控制打标点位与样式 {y_col: {"idx":..., "offset":(dx,dy), "ha":..., "va":...}}
    fmt: str = "{y:.1f}%",               # 数据标签格式化（若percent=True则传入 y*100）
    percent: bool = True,                # True: y当作0~1比例；False: 原值
    fontsize: int = 14,                  # 基础字号
    dpi: int = 300,
    save_dir: str = None,
    suffix: str = ".png",
    bbox_inches: str = "tight",
    y_lim: tuple = None,                 # (ymin, ymax)；None则自动
    auto_y_pad: float = 1.15,            # 自动y上边距系数
    spine_off: tuple = ("top", "right", "left"),
    hide_y_ticks: bool = True,
    hide_y_tick_labels: bool = True,
):
    """
    - 画多折线图（统一为 16:9 大图，适配PPT），并为每个点添加数据标签，同时保存到 imgs 目录（或指定 save_dir），返回图片路径。

    Parameters
    ----------
    - df : pd.DataFrame 数据源表。至少包含 x_col + y_cols。
    - x_col : str x轴列名，函数内部会转成字符串并按字典序排序后绘制。
    - y_cols : list 需要绘制的多条折线列名列表。缺列会自动补 NaN（不报错）。
    - title : str 图标题，同时作为输出图片文件名（会做简单的 / 和 \\ 替换）。

    Keyword Args
    ------------
    - labels : dict, default None
        显示名映射：{列名: 图例显示名}。不传则用列名本身。
    - colors : dict, default None
        颜色映射：{列名: '#RRGGBB'}。不传则使用函数内置默认调色盘。
    - linestyles : dict, default None
        线型映射：{列名: '-', '--', ':', '-.'}。不传则默认 '-' ,常用：{'总计': ':'} 让总计虚线。
    - markers : dict, default None
        点型映射：{列名: 'o', 's', '^', ...}。不传默认 'o'。
    - offsets : dict, default None
        数据标签偏移：{列名: (dx, dy)}，单位 points。
        dy>0 标签在点上方；dy<0 在下方；用于避免标签重叠。
        未指定的列默认 (0, 8)。
    - fmt : str, default "{y:.1f}%"
        数据标签字符串格式。会使用 fmt.format(y=shown)。
        - percent=True 时 shown = 原值*100
        - percent=False 时 shown = 原值
    - percent : bool, default True
        True：把 y 当作 0~1 的比例，并自动把标签显示为百分比（乘以100）。
        False：y 按原值绘制与标注。
    - fontsize : int, default 14
        基础字号：x刻度/图例用 fontsize；点标签用 fontsize-2；标题用 fontsize+6。
    - dpi : int, default 300
        保存图片DPI。
    - save_dir : str, default None
        保存目录。None 时默认 os.getcwd()/imgs。
    - suffix : str, default ".png"
        输出图片后缀。
    - bbox_inches : str, default "tight"
        fig.savefig 的 bbox_inches 参数（tight 可减少白边）。
    - legend_ncol : int, default None
        图例列数。None 时自动 min(len(y_cols), 6)。
    - y_lim : tuple, default None
        y轴范围 (ymin, ymax)。None 时自动计算并按 auto_y_pad 留上边距。
        percent=True 时自动把 ymin 固定为 0。
    - auto_y_pad : float, default 1.15
        自动y上边距系数（只在 y_lim=None 时生效）。
    - spine_off : tuple, default ("top","right","left")
        需要隐藏的边框集合。若要保留底框线，请不要包含 "bottom"。
    - hide_y_ticks : bool, default True
        True：隐藏 y 轴刻度线（length=0）。
    - hide_y_tick_labels : bool, default True
        True：隐藏 y 轴刻度文字（更“干净”的PPT风格）。
    - label_cfg = {
      "某列": {"idx": "min"/"max"/-1/0/[0,2,4]/"all"/[], "offset":(dx,dy), "ha":"center", "va":"top"}
    """

    if df is None or len(df) == 0:
        raise ValueError("df 为空，无法绘图。")

    # 默认保存目录
    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), "imgs")
    os.makedirs(save_dir, exist_ok=True)

    # 默认配色（6个）
    default_palette = [
        "#2F6EEB",  # 蓝
        "#F39C12",  # 橙
        "#9e9e9e",  # 灰
        "#F1C40F",  # 黄
        "#34B4F4",  # 浅蓝
        "#6A5ACD",  # 紫 
    ]

    d = df.copy()
    if x_col not in d.columns:
        if d.index.name == x_col:
            d = d.reset_index()
        else:
            raise KeyError(f"x_col='{x_col}' 不在 df.columns，且 df.index.name != '{x_col}'。请先 reset_index() 或传入正确的 x_col。")

    d[x_col] = d[x_col].astype(str)
    d = d.sort_values(x_col)

    # x轴：严格按顺序
    x_labels = d[x_col].tolist()
    x = np.arange(len(d))

    # 颜色映射：默认 -> 用户覆盖
    color_map = {c: default_palette[i % len(default_palette)] for i, c in enumerate(y_cols)}
    if colors:
        color_map.update(colors)

    linestyles = linestyles or {}
    markers = markers or {}
    offsets = offsets or {}
    labels = labels or {}
    aligns = aligns or {}
    label_cfg = label_cfg or {}

    # offsets 也支持 list/tuple 简写（兼容你原来用法）
    if isinstance(offsets, (list, tuple)):
        if len(offsets) != len(y_cols):
            raise ValueError(f"offsets 长度({len(offsets)})必须等于 y_cols 长度({len(y_cols)})")
        offsets = {c: off for c, off in zip(y_cols, offsets)}

    def _resolve_label_indices(y: np.ndarray, idx_rule):
        finite_idx = [i for i, v in enumerate(y) if np.isfinite(v)]
        if not finite_idx:
            return []

        if idx_rule is None or idx_rule == "all":
            return finite_idx

        if idx_rule == "min":
            yy = np.where(np.isfinite(y), y, np.nan)
            return [int(np.nanargmin(yy))] if np.isfinite(yy).any() else []
        if idx_rule == "max":
            yy = np.where(np.isfinite(y), y, np.nan)
            return [int(np.nanargmax(yy))] if np.isfinite(yy).any() else []

        if isinstance(idx_rule, (int, np.integer)):
            idx_list = [int(idx_rule)]
        elif isinstance(idx_rule, (list, tuple, set)):
            idx_list = [int(i) for i in idx_rule]
        else:
            raise ValueError(f"idx 只支持 None/'all'/[]/int/list/'min'/'max'，当前为: {type(idx_rule)}")

        out = []
        for i in idx_list:
            if i == -1:
                out.append(finite_idx[-1])
            elif 0 <= i < len(y) and np.isfinite(y[i]):
                out.append(i)

        # 去重保持顺序
        seen = set()
        out2 = []
        for i in out:
            if i not in seen:
                out2.append(i)
                seen.add(i)
        return out2

    fig, ax = plt.subplots(figsize=(13.33, 7.5))

    plotted_any = False
    y_all = []

    for c in y_cols:
        if c not in d.columns:
            d[c] = np.nan

        y = pd.to_numeric(d[c], errors="coerce").astype(float).values
        y_all.append(y)

        # 画线（确保有 label）
        ax.plot(
            x, y,
            label=labels.get(c, c),
            color=color_map.get(c),
            marker=markers.get(c, "o"),
            linewidth=2.5,
            linestyle=linestyles.get(c, "-"),
        )
        plotted_any = True

        # ---------- 打标策略（默认全标；可用 label_cfg 指定） ----------
        cfg = label_cfg.get(c, {}) if isinstance(label_cfg, dict) else {}

        # offset：label_cfg 优先，其次 offsets
        dx, dy = cfg.get("offset", offsets.get(c, (0, 8)))

        # ha/va：label_cfg 优先，其次 aligns（再按 dy 推断 va）
        ha = "center"
        va = "bottom" if dy >= 0 else "top"

        # aligns 兼容 dict / tuple
        a = aligns.get(c, None)
        if isinstance(a, dict):
            if a.get("ha") is not None:
                ha = a["ha"]
            if a.get("va") is not None:
                va = a["va"]
        elif isinstance(a, (list, tuple)) and len(a) == 2:
            if a[0] is not None:
                ha = a[0]
            if a[1] is not None:
                va = a[1]

        if cfg.get("ha") is not None:
            ha = cfg["ha"]
        if cfg.get("va") is not None:
            va = cfg["va"]

        idx_rule = cfg.get("idx", "all")
        idxs_to_label = _resolve_label_indices(y, idx_rule)

        for i in idxs_to_label:
            yi = y[i]
            if not np.isfinite(yi):
                continue
            shown = (yi * 100.0) if percent else yi
            ax.annotate(
                fmt.format(y=shown),
                xy=(x[i], yi),
                xytext=(dx, dy),
                textcoords="offset points",
                ha=ha,
                va=va,
                fontsize=fontsize,
                color="black",
                fontweight="bold",
                annotation_clip=True,
            )

    if not plotted_any:
        raise ValueError("y_cols 没有可绘制列。")

    ax.set_title(title, fontsize=fontsize + 6, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=0, fontsize=fontsize)

    ax.set_ylabel("")
    if hide_y_ticks:
        ax.tick_params(axis="y", length=0)
    if hide_y_tick_labels:
        ax.set_yticklabels([])

    # y范围：自动留白 or 指定
    if y_lim is not None:
        ax.set_ylim(*y_lim)
    else:
        y_concat = np.concatenate([v[np.isfinite(v)] for v in y_all if v is not None and np.any(np.isfinite(v))]) \
            if any(np.any(np.isfinite(v)) for v in y_all) else np.array([0.0])
        ymin = float(np.nanmin(y_concat)) if len(y_concat) else 0.0
        ymax = float(np.nanmax(y_concat)) if len(y_concat) else 1.0

        # 若是比例，默认从0开始更符合展示习惯
        if percent:
            ymin = 0.0

        # 上边距留白
        ymax = ymax * auto_y_pad if ymax > 0 else 1.0
        ax.set_ylim(ymin, ymax)
    
    # 去边框（保持一致风格）
    for sp in spine_off:
        ax.spines[sp].set_visible(False)

    # 图例放底部居中
    ax.legend(ncol=len(y_cols), loc="lower center", bbox_to_anchor=(0.5, -0.15), frameon=False, fontsize=14)
    plt.tight_layout()

    safe_name = title.replace("/", "_").replace("\\", "_").strip()
    img_path = os.path.join(save_dir, f"{safe_name}{suffix}")
    fig.savefig(img_path, dpi=dpi, bbox_inches=bbox_inches)
    plt.show()
    plt.close(fig)


### PPT绘图函数演示

#### 1.1 老客月发起订单数（万）

In [ ]:
# 发起 & 资金通过率
query='''
WITH apply_risk_score AS
(
	SELECT  a.*
	       ,CASE WHEN b.model_value IS NULL OR b.model_value < 0 THEN '空'
	             WHEN b.model_value < 2 THEN 'A'
	             WHEN b.model_value < 3 THEN 'B'
	             WHEN b.model_value < 5 THEN 'CD'
	             WHEN b.model_value < 7 THEN 'EF'
	             WHEN b.model_value < 9 THEN 'GH'  ELSE 'IJK' END AS apply_model_score_category
	FROM
	(
		SELECT  first_order_number
		       ,first_order_time
		       ,risk_status
		       ,loan_status
		       ,loan_time
		       ,cust_no
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE 1 = 1
		AND pt = '{bizdate}'
		AND app IN ('xyf01', 'fxk')
		AND business_line IN ('APP', '小程序端')
		AND loan_flag IN ('复贷', '加贷')
		AND DATE(first_order_time) >= '{begin_date}'
		AND DATE(first_order_time) <= '{end_date}'
	)a --每次更新一下日期条件 
	LEFT JOIN xyf_dwd.dwd_risk_model_b_card_df b
	ON a.cust_no = b.cust_no AND b.pt >= '20240901' AND DATE(a.first_order_time) = DATE(b.decision_time)
)

SELECT  substr(first_order_time,1,7)                                                                                  AS apply_month
       ,apply_model_score_category
       ,COUNT(DISTINCT first_order_number)                                                                            AS order_cnt
       ,COUNT(DISTINCT CASE WHEN risk_status = 'pass' THEN first_order_number END)                                    AS risk_pass_order_cnt
       ,COUNT(DISTINCT CASE WHEN risk_status = 'pass' THEN first_order_number END)/COUNT(DISTINCT first_order_number) AS risk_pass_rate
       ,COUNT(DISTINCT CASE WHEN loan_status = 'success' THEN first_order_number END)/COUNT(DISTINCT CASE WHEN risk_status = 'pass' THEN first_order_number END) AS fund_pass_rate
       ,COUNT(DISTINCT CASE WHEN loan_status = 'success' AND (unix_timestamp(loan_time) - unix_timestamp(first_order_time)) BETWEEN 0 AND 3600*2 THEN first_order_number END)/COUNT(DISTINCT CASE WHEN risk_status = 'pass' THEN first_order_number END) AS fund_2h_pass_rate
FROM apply_risk_score
GROUP BY  substr(first_order_time,1,7)
         ,apply_model_score_category;
'''.format(bizdate=bizdate , begin_date=begin_date, end_date=end_date)

data = o.execute_sql(query,  
    hints={
        'odps.sql.submit.mode': 'script',                     # 多语句
        'odps.sql.validate.orderby.limit': 'false'            # 关闭 ORDER BY 无 LIMIT 校验
    }).open_reader().to_pandas()

data["apply_month"] = data["apply_month"].astype(str)
data['apply_model_score_category'] = data['apply_model_score_category'].astype(str)
data["order_cnt"] = data["order_cnt"].astype(float)
data["risk_pass_order_cnt"] = data["risk_pass_order_cnt"].astype(float)
data["risk_pass_rate"] = data["risk_pass_rate"].astype(float)
data["fund_pass_rate"] = data["fund_pass_rate"].astype(float)
data["fund_2h_pass_rate"] = data["fund_2h_pass_rate"].astype(float)
# 丢掉apply_model_score_category为空的数据
clean_data = data[data['apply_model_score_category'] != '空']

In [ ]:
pivot_order_cnt = (
    clean_data.pivot_table(
        index="apply_month",
        columns='apply_model_score_category',
        values='order_cnt',
        aggfunc="sum",
        fill_value=0.0,
    )
    .sort_index()
) 
pivot_order_cnt ["总计"] = pivot_order_cnt.sum(axis=1)

plot_stacked_bar_from_pivot(
    pivot=pivot_order_cnt/1e4, #转化为万
    cols=["A", "B", "CD", "EF", "GH", "IJK"],
    title="老客月发起订单数（万）",
    p=0.5,               
)

#### 1.2 老客月风险（订单）通过率

In [ ]:
pivot_risk_pass_order_cnt = (
    clean_data.pivot_table(
        index="apply_month",
        columns="apply_model_score_category",
        values="risk_pass_order_cnt",
        aggfunc="sum",
        fill_value=0.0,
    )
    .sort_index()
)
pivot_risk_pass_order_cnt["总计"] = pivot_risk_pass_order_cnt.sum(axis=1)

agg_pct = pivot_risk_pass_order_cnt.div(pivot_order_cnt.replace(0, np.nan)).fillna(0.0)
agg_pct = agg_pct.sort_index().tail(12) # 数据多于12个月，只保留最新12个月

plot_lines_from_df(
    df=agg_pct,
    x_col="apply_month",
    y_cols=["A", "B", "CD", "EF", "GH", "IJK", "总计"],
    title="老客月风险（订单）通过率",
    linestyles = {"总计": ":"},  # 总计设为虚线
    colors={"总计": "#111111"},
    fmt="{y:.1f}%",
    percent=True,
    y_lim = (0,1.0),
    label_cfg={
        "A": {"idx":'all', "offset": (0, 10),  "va": "bottom" , "ha" : "center"},
        "B": {"idx":'all', "offset": (0, 10),  "va": "bottom" , "ha" : "center"},
        "CD": {"idx": 'all', "offset": (0, 5),   "va": "bottom", "ha" : "center"},
        "EF": {"idx": 'all', "offset": (0, 5),  "va": "bottom", "ha" : "center"},
        "GH": {"idx": [0,1,2,3,4,5,6], "offset": (0, -5),  "va": "top", "ha" : "center"},
        "IJK":  {"idx": [0,1,2,3,4,5,6], "offset": (0, -5), "va": "top", "ha" : "center"},
        "总计": {"idx": 'all', "offset": (0, 5), "va": "bottom", "ha" : "center"},
    }
)

#### 1.3 老客月资金通过率


In [ ]:
pivot_fund_pass_rate = (
    clean_data.pivot_table(
        index="apply_month",
        columns="apply_model_score_category",
        values="fund_pass_rate",
        aggfunc="mean",
        fill_value=0.0,
    )
    .sort_index()
)
pivot_fund_pass_rate["总计"] = pivot_fund_pass_rate.mean(axis=1)

plot_lines_from_df(
    df=pivot_fund_pass_rate,
    x_col="apply_month",
    y_cols=["A", "B", "CD", "EF", "GH", "IJK", "总计"],
    title="老客月资金通过率",
    colors={"总计": "#111111"},
    linestyles = {"总计": ":"},  # 总计设为虚线
    fmt="{y:.1f}%",
    percent=True,
    y_lim = (0.6,1.1),
    label_cfg={
        "A": {"idx":'all', "offset": (0, 10),  "va": "bottom"},
        "B": {"idx": [4], "offset": (0, -10),  "va": "bottom"},
        "CD": {"idx": [4], "offset": (0, 10),   "va": "bottom"},
        "EF": {"idx": [4], "offset": (0, 10),  "va": "top"},
        "GH": {"idx": [4], "offset": (0, -5),  "va": "top"},
        "IJK":  {"idx": [4], "offset": (0, -2), "va": "top"},
        "总计": {"idx": [4], "offset": (0, 0), "va": "bottom"},
    }
)


#### 1.4 老客月资金通过率（申请2小时内）


In [ ]:
pivot_fund_2h_pass_rate = (
    clean_data.pivot_table(
        index="apply_month",
        columns="apply_model_score_category",
        values="fund_2h_pass_rate",
        aggfunc="mean",
        fill_value=0.0,
    )
    .sort_index()
)
pivot_fund_2h_pass_rate["总计"] = pivot_fund_2h_pass_rate.mean(axis=1)

plot_lines_from_df(
    df=pivot_fund_2h_pass_rate,
    x_col="apply_month",
    y_cols=["A", "B", "CD", "EF", "GH", "IJK", "总计"],
    title="老客月资金通过率（申请2小时内）",
    colors={"总计": "#111111"},
    linestyles = {"总计": ":"},  # 总计设为虚线
    fmt="{y:.1f}%",
    percent=True,
    y_lim = (0.5,1.1),
    label_cfg={
        "A": {"idx":'all', "offset": (0, 10),  "va": "bottom"},
        "B": {"idx": [4], "offset": (0, -10),  "va": "bottom"},
        "CD": {"idx": [4], "offset": (0, -10),   "va": "bottom"},
        "EF": {"idx": [4], "offset": (0, 0),  "va": "top"},
        "GH": {"idx": [4], "offset": (0, 0),  "va": "top"},
        "IJK":  {"idx": [4], "offset": (0, 0), "va": "top"},
        "总计": {"idx": [4], "offset": (0, 10), "va": "bottom"},
    }
)


### PPT图片替换

In [ ]:
def replace_pictures_by_name_batch(
    ppt_path,
    img_dir,
    out_path=None,
    suffix=".png",
    dry_run=False,        # 是否试运行
    allowed_types=None,   # 例如 {MSO_SHAPE_TYPE.PICTURE, MSO_SHAPE_TYPE.CHART}
    strict=False,         # True：图片缺失就报错；False：跳过
):
    """
    批量替换：图片文件名(不含后缀) = PPT 形状名称（shape.name）。

    - 会遍历 PPT 全部页的 shapes
    - 若 shape.name 在 img_dir 中存在同名图片，则删除该 shape 并按原 left/top/width/height 插入图片
    - dry_run=True：试运行，只打印“将要替换哪些 shape、用哪张图片替换”，但不删除 shape、不插入图片、不保存文件
    - allowed_types 为 None 时不限制 shape 类型；否则仅处理指定类型（如 CHART / PICTURE）
    - strict=True 时：遇到有匹配形状但找不到图片会抛错（方便做完整性校验）
    """
    assert os.path.isfile(ppt_path), f"PPT 不存在: {ppt_path}"
    assert os.path.isdir(img_dir), f"图片目录不存在: {img_dir}"

    # 建立：name -> img_path（同名覆盖取最后一个）
    files = {}
    for fn in os.listdir(img_dir):
        if not fn.lower().endswith(suffix.lower()):
            continue
        stem = os.path.splitext(fn)[0].strip()
        if stem:
            files[stem] = os.path.join(img_dir, fn)

    if not files:
        print(f"目录内无 {suffix} 图片：{img_dir}")
        return {"hit": 0, "miss": 0, "total_imgs": 0, "total_shapes": 0}

    prs = Presentation(ppt_path)

    hit, miss, skipped = 0, 0, 0
    total_shapes = 0

    # 收集PPT里“可参与匹配”的shape名
    ppt_names = set()

    # 遍历每页每个 shape
    for si, slide in enumerate(prs.slides):
        # list()：因为下面会 remove，避免迭代器被破坏
        for shp in list(slide.shapes):
            total_shapes += 1
            name = getattr(shp, "name", "").strip()
            if not name:
                continue

            # 类型过滤（可选）
            if allowed_types is not None and shp.shape_type not in allowed_types:
                skipped += 1
                continue
            ppt_names.add(name)
            
            # 仅当存在同名图片时才替换
            img_path = files.get(name)
            if not img_path:
                miss += 1
                continue

            left, top, width, height = shp.left, shp.top, shp.width, shp.height
            print(f"[替换] 第{si+1}页《{name}》 type={shp.shape_type} -> {img_path}")

            if dry_run:
                hit += 1
                continue

            # 先删除原 shape
            slide.shapes._spTree.remove(shp._element)

            # 再插入图片并保持尺寸 + 恢复 name
            pic = slide.shapes.add_picture(img_path, left, top)
            pic.width, pic.height = width, height

            # 关键：把新图片 shape 的 name 改回原来的 name（保持映射逻辑）
            try:
                pic.name = name
            except Exception as e:
                # 极少数情况下会失败，但一般都可用
                print(f"[警告] 无法设置图片名称为 {name}（第{si+1}页）：{e}")

            hit += 1

    # ✅ 直接打印：文件夹里没匹配上的图片名
    img_only_names = sorted(set(files.keys()) - ppt_names)
    print("\n========== 文件夹里没匹配上的图片（img_only_names）==========")
    if img_only_names:
        for n in img_only_names:
            print(n)
    else:
        print("无（全部图片都能在PPT里找到同名shape）")
    print("===========================================================\n")

    # strict 校验：如果要求“形状有名字的都必须有图”，可在这里加强
    if strict:
        # 找出 img_dir 里有图但PPT里找不到同名shape，也可以加校验；此处仅对“命名shape找不到图”提示
        if miss > 0:
            raise ValueError(f"strict=True：存在 {miss} 个命名shape未找到对应图片（suffix={suffix}）。")

    if not dry_run:
        save_path = out_path or ppt_path
        prs.save(save_path)
        print(f"已保存到：{save_path}")

    print(f"批量替换完成：命中 {hit}，未命中(无同名图片) {miss}，跳过(类型不符) {skipped}，候选图片 {len(files)}，遍历shape {total_shapes}。")
    return {"hit": hit, "miss": miss, "skipped": skipped, "total_imgs": len(files), "total_shapes": total_shapes}


# 用法示例：把 imgs 目录里 “形状名.png” 批量替换到 PPT
from pptx.enum.shapes import MSO_SHAPE_TYPE
replace_pictures_by_name_batch(
    ppt_path=ppt_path,
    img_dir=os.path.join(os.getcwd(), "imgs"),
    out_path=out_path,
    suffix=".png",
    dry_run=False,
    allowed_types={MSO_SHAPE_TYPE.CHART, MSO_SHAPE_TYPE.PICTURE},  # 只替换图表/图片占位
    strict=False,
)